# 13 — Comparar modelos de embeddings con la configuración final

Medimos si otro modelo de embeddings recupera mejor que `nomic-embed-text` con la configuración final del RAG: troceo `trocear_por_estructura` 1500/200 sin ruta, 20 candidatos, reranker `cross-encoder/ms-marco-MiniLM-L-6-v2`, k=4, métrica de varias evidencias y los seis metadatos de #26. Son seis modelos (nomic, `qwen3_06b`, `qwen3_8b`, `bge_m3`, `embeddinggemma`, `jina_code`) en cuatro configuraciones sobre las 32 preguntas A-D: sin reranker / con reranker × sin filtro / filtro perfecto por `modulo` con repetición (umbral 0,0).

## Criterio de decisión (RF-13), fijado antes de medir

- Un modelo sustituye a nomic si, con reranker y filtro `modulo` con repetición, suma al menos 2 aciertos ev_hit@4 sobre nomic (≥ 30/32) sin bajar hit@4 por fuente (≥ 31/32).
- Si varios cumplen, gana el de más ev_hit@4; si empatan, el de menor latencia media de consulta en esa configuración.
- La latencia de consulta y el tiempo de indexación se documentan.

Comprobación de la referencia (RF-11), antes de guardar: tras medir, nomic debe reproducir, pregunta a pregunta, `filtros_metadatos_por_pregunta.csv`: «con reranker, sin filtro» igual a «1 sin filtro» (30/32 fuente, 26/32 evidencia) y «con reranker, filtro modulo con repeticion» igual a «3 perfecto modulo con repeticion» (31/32 fuente, 28/32 evidencia). Si no, el notebook imprime las diferencias y para sin guardar.

Todas las configuraciones se miden en el mismo equipo y con el reranker en el mismo dispositivo (RNF-07). Sus latencias no se comparan con las de CSV anteriores medidos con el reranker en CPU.

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

In [1]:
import json
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
import torch
from qdrant_client import QdrantClient
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, leer_evidencias, leer_golden_set,
    rango_primer_acierto, rango_primer_acierto_evidencia,
)
from consultar_rag import RERANKER, ConsultorRAG
from embeddings_modelos import (
    cargar_codificador, dispositivo, dispositivo_ollama, elegir_modelo,
    liberar_vram_ollama, nombre_coleccion,
)
from metadatos import modulo_de_fichero

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
REGISTRO = RAIZ / "data" / "resultados" / "indexacion_embeddings.json"
REFERENCIA_CSV = RAIZ / "data" / "resultados" / "filtros_metadatos_por_pregunta.csv"
SALIDA_RESUMEN = RAIZ / "data" / "resultados" / "modelos_embedding.csv"
SALIDA_PREGUNTA = RAIZ / "data" / "resultados" / "modelos_embedding_por_pregunta.csv"
CANDIDATOS = 20
K = 4
UMBRAL = 0.0
N_CHUNKS = 7955
# Orden de medición: Ollama primero y jina_code el último, tras liberar la VRAM de Ollama
MODELOS_ORDEN = ["nomic", "qwen3_06b", "qwen3_8b", "bge_m3", "embeddinggemma", "jina_code"]
# (nombre, reordenar, filtro perfecto por modulo)
CONFIGURACIONES = [
    ("sin reranker, sin filtro", False, False),
    ("sin reranker, filtro modulo con repeticion", False, True),
    ("con reranker, sin filtro", True, False),
    ("con reranker, filtro modulo con repeticion", True, True),
]
print("Modelos:", ", ".join(MODELOS_ORDEN))

Modelos: nomic, qwen3_06b, qwen3_8b, bge_m3, embeddinggemma, jina_code


## 3. Dispositivo

`dispositivo()` (cuda, mps o cpu, por ese orden) decide dónde corren el reranker y `jina_code`. Ollama decide por su cuenta dónde corre cada modelo; su dispositivo se lee de `ps()` tras la consulta de calentamiento. `qwen3_8b` no cabe entera en la GPU y sale `cuda+cpu`, así que su latencia no se mide en GPU completa y no es comparable con la de los demás.

In [2]:
DISPOSITIVO = dispositivo()
print("Dispositivo:", DISPOSITIVO)
print("torch:", torch.__version__)
if DISPOSITIVO == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Dispositivo: cuda
torch: 2.14.1+cu130
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## 4. Comprobación de colecciones y registro (RF-10)

Cada modelo necesita su colección con 7 955 puntos, con metadatos, y su entrada en `indexacion_embeddings.json`. Si falta algo, el notebook para y nombra la orden que lo crea.

In [3]:
qdrant = QdrantClient("localhost", port=6333)
registro = json.loads(REGISTRO.read_text(encoding="utf-8"))
faltan = []
for clave in MODELOS_ORDEN:
    coleccion = nombre_coleccion(clave)
    if clave == "nomic":
        orden = "uv run python scripts/indexar_embeddings.py --referencia"
    else:
        orden = f"uv run python scripts/indexar_embeddings.py {clave}"
    try:
        puntos = qdrant.count(coleccion, exact=True).count
        primero, _ = qdrant.scroll(coleccion, limit=1, with_payload=True, with_vectors=False)
        con_metadatos = bool(primero) and "modulo" in (primero[0].payload or {})
    except Exception:
        puntos, con_metadatos = 0, False
    if puntos != N_CHUNKS or not con_metadatos or clave not in registro:
        faltan.append((clave, orden))
for clave, orden in faltan:
    print(f"FALTA {clave}: ejecuta `{orden}`")
assert not faltan, "Faltan colecciones o entradas del registro: no se mide nada"
pd.DataFrame(registro).T[["nombre", "proveedor", "dimension", "num_ctx", "truncados", "tiempo_indexacion_s", "dispositivo"]]

,nombre,proveedor,dimension,num_ctx,truncados,tiempo_indexacion_s,dispositivo
nomic,nomic-embed-text,ollama,768,2048,42,228.249303,cuda
qwen3_06b,qwen3-embedding:0.6b,ollama,1024,8192,2,261.373008,cuda
embeddinggemma,embeddinggemma,ollama,768,2048,47,247.365418,cuda
qwen3_8b,qwen3-embedding:8b,ollama,4096,8192,2,1573.431029,cuda+cpu
jina_code,jinaai/jina-code-embeddings-1.5b,sentence-transformers,1536,8192,2,1814.871073,cuda
bge_m3,bge-m3,ollama,1024,8192,3,276.259339,cuda


## 5. Golden set y evidencias

In [4]:
preguntas = [p for p in leer_golden_set(GOLDEN) if p["categoria"] != "E"]
evidencias = leer_evidencias(EVIDENCIAS)
print("Preguntas A-D:", len(preguntas))

Preguntas A-D: 32


## 6. Filtro perfecto por `modulo`

Misma regla que el notebook 12: el filtro perfecto de cada pregunta son los `modulo` no vacíos de sus fuentes esperadas, sin repetidos. Sin filtro perfecto, la pregunta va sin filtro en las configuraciones con filtro.

In [5]:
def sin_repetidos(valores):
    return list(dict.fromkeys(v for v in valores if v))

perfecto_modulo = [sin_repetidos(modulo_de_fichero(CARPETA_CORPUS, f) for f in p["fuentes"]) for p in preguntas]
print("Preguntas con filtro perfecto por modulo:", sum(bool(v) for v in perfecto_modulo), "de", len(preguntas))

Preguntas con filtro perfecto por modulo: 22 de 32


## 7. Reranker

Se carga una vez, en el dispositivo de la sección 3, y lo usan todos los modelos.

In [6]:
reranker = CrossEncoder(RERANKER, device=DISPOSITIVO)
DISPOSITIVO_RERANKER = str(next(reranker.model.parameters()).device.type)
print("Reranker:", RERANKER, "en", DISPOSITIVO_RERANKER)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2 en cuda


## 8. Medición

Una fila por modelo, configuración y pregunta. Por modelo, una consulta de calentamiento sin medir fija `dispositivo_embeddings`: en los modelos de Ollama sale de `ps()` (`qwen3_8b` puede salir `cuda+cpu`, con latencia no medida en GPU completa) y en `jina_code` de `codificador.device.type`. Los modelos de Ollama van primero; antes de `jina_code` se libera la VRAM de Ollama (RF-19), de modo que nunca coinciden `qwen3_8b` y `jina_code` en memoria.

In [7]:
cliente = ollama.Client()
filas = []
dispositivo_embeddings = {}
for clave in MODELOS_ORDEN:
    nombre = MODELOS[clave]["nombre"]
    codificador = None
    if MODELOS[clave]["proveedor"] == "ollama":
        consultor = ConsultorRAG(qdrant, cliente, reranker, nombre_coleccion(clave), clave)
        consultor.consultar(preguntas[0]["query"])  # calentamiento
        dispositivo_embeddings[clave] = dispositivo_ollama(cliente, nombre)
    else:
        liberar_vram_ollama(cliente)
        codificador = cargar_codificador(clave)
        consultor = ConsultorRAG(qdrant, cliente, reranker, nombre_coleccion(clave), clave, codificador)
        consultor.consultar(preguntas[0]["query"])  # calentamiento
        dispositivo_embeddings[clave] = codificador.device.type
    print(clave, "->", dispositivo_embeddings[clave])
    for config, reordenar, con_filtro in tqdm(CONFIGURACIONES, desc=clave):
        for i, pregunta in enumerate(preguntas):
            valor = perfecto_modulo[i] if con_filtro else []
            filtros = {"modulo": valor} if valor else None
            inicio = time.perf_counter()
            resultados = consultor.consultar(
                pregunta["query"], candidatos=CANDIDATOS, k=K, filtros=filtros,
                umbral=UMBRAL, reordenar=reordenar,
            )
            latencia_ms = (time.perf_counter() - inicio) * 1000
            rango = rango_primer_acierto([r["fuente"] for r in resultados], pregunta["fuentes"])
            rango_ev = rango_primer_acierto_evidencia(resultados, evidencias[pregunta["id"]])
            filas.append({
                "modelo": clave,
                "configuracion": config,
                "id": pregunta["id"],
                "categoria": pregunta["categoria"],
                "filtro": "|".join(valor) if filtros else "",
                "repetida_sin_filtro": bool(filtros) and not any(r["filtrado"] for r in resultados),
                "rango fuente": rango,
                "rango evidencia": rango_ev,
                "hit@4": rango is not None and rango <= K,
                "ev_hit@4": rango_ev is not None and rango_ev <= K,
                "latencia total (ms)": latencia_ms,
            })
    del consultor, codificador
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()
    elif DISPOSITIVO == "mps":
        torch.mps.empty_cache()
por_pregunta = pd.DataFrame(filas)
print(len(por_pregunta), "filas")

nomic -> cuda


nomic:   0%|          | 0/4 [00:00<?, ?it/s]

qwen3_06b -> cuda


qwen3_06b:   0%|          | 0/4 [00:00<?, ?it/s]

qwen3_8b -> cuda+cpu


qwen3_8b:   0%|          | 0/4 [00:00<?, ?it/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

jina_code -> cuda


jina_code:   0%|          | 0/4 [00:00<?, ?it/s]

768 filas


## 9. Comprobación de la referencia nomic (RF-11)

Se compara pregunta a pregunta con `filtros_metadatos_por_pregunta.csv`. Si alguna pregunta no coincide, se imprime la diferencia y la celda para sin guardar nada.

In [8]:
anterior = pd.read_csv(REFERENCIA_CSV)
pares = [
    ("con reranker, sin filtro", "1 sin filtro", (30, 26)),
    ("con reranker, filtro modulo con repeticion", "3 perfecto modulo con repeticion", (31, 28)),
]
diferencias = []
for config, config_anterior, esperado_n in pares:
    nuevo = por_pregunta[(por_pregunta["modelo"] == "nomic") & (por_pregunta["configuracion"] == config)].set_index("id")
    viejo = anterior[anterior["configuracion"] == config_anterior].set_index("id")
    for id_, fila in nuevo.iterrows():
        for columna in ("rango fuente", "rango evidencia"):
            esperado, obtenido = viejo.loc[id_, columna], fila[columna]
            if not (pd.isna(esperado) and pd.isna(obtenido)) and esperado != obtenido:
                diferencias.append((config, id_, columna, esperado, obtenido))
    n_fuente, n_ev = int(nuevo["hit@4"].sum()), int(nuevo["ev_hit@4"].sum())
    print(f"{config}: {n_fuente}/32 fuente, {n_ev}/32 evidencia (referencia {esperado_n[0]}/32 y {esperado_n[1]}/32)")
    if (n_fuente, n_ev) != esperado_n:
        diferencias.append((config, "total", "cifras", esperado_n, (n_fuente, n_ev)))
for d in diferencias:
    print("DIFERENCIA", *d)
print("Diferencias con filtros_metadatos_por_pregunta.csv:", len(diferencias))
assert not diferencias, "RF-11 no se cumple: no se guarda nada"

con reranker, sin filtro: 30/32 fuente, 26/32 evidencia (referencia 30/32 y 26/32)
con reranker, filtro modulo con repeticion: 31/32 fuente, 28/32 evidencia (referencia 31/32 y 28/32)
Diferencias con filtros_metadatos_por_pregunta.csv: 0


## 10. Métricas por modelo y configuración

In [9]:
def metrica(df, columna):
    rangos = [None if pd.isna(v) else int(v) for v in df[columna]]
    return calcular_metricas(rangos, K)

resumen_filas = []
for clave in MODELOS_ORDEN:
    for config, reordenar, _ in CONFIGURACIONES:
        df = por_pregunta[(por_pregunta["modelo"] == clave) & (por_pregunta["configuracion"] == config)]
        fuente, evidencia = metrica(df, "rango fuente"), metrica(df, "rango evidencia")
        resumen_filas.append({
            "modelo": clave,
            "configuracion": config,
            "hit@1": fuente["hit@1"],
            "hit@4": fuente["hit@4"],
            "MRR": fuente["mrr"],
            "ev_hit@1": evidencia["hit@1"],
            "ev_hit@4": evidencia["hit@4"],
            "ev_MRR": evidencia["mrr"],
            "latencia media consulta (ms)": df["latencia total (ms)"].mean(),
            "tiempo indexacion (s)": registro[clave]["tiempo_indexacion_s"],
            "num_ctx": registro[clave]["num_ctx"],
            "truncados": registro[clave]["truncados"],
            "dimension": registro[clave]["dimension"],
            "dispositivo_embeddings": dispositivo_embeddings[clave],
            "dispositivo_reranker": DISPOSITIVO_RERANKER if reordenar else "",
        })
resumen = pd.DataFrame(resumen_filas)
resumen

,modelo,configuracion,hit@1,hit@4,MRR,ev_hit@1,ev_hit@4,ev_MRR,latencia media consulta (ms),tiempo indexacion (s),num_ctx,truncados,dimension,dispositivo_embeddings,dispositivo_reranker
0,nomic,"sin reranker, sin filtro",0.71875,0.90625,0.786458,0.43750,0.65625,0.528646,96.344134,228.249303,2048,42,768,cuda,
1,nomic,"sin reranker, filtro modulo con repeticion",0.90625,0.96875,0.937500,0.53125,0.71875,0.614583,61.904419,228.249303,2048,42,768,cuda,
2,nomic,"con reranker, sin filtro",0.68750,0.93750,0.796875,0.43750,0.81250,0.604167,109.960237,228.249303,2048,42,768,cuda,cuda
3,nomic,"con reranker, filtro modulo con repeticion",0.84375,0.96875,0.906250,0.56250,0.87500,0.700521,105.693175,228.249303,2048,42,768,cuda,cuda
4,qwen3_06b,"sin reranker, sin filtro",0.65625,0.96875,0.791667,0.43750,0.71875,0.557292,61.670653,261.373008,8192,2,1024,cuda,
5,qwen3_06b,"sin reranker, filtro modulo con repeticion",0.87500,0.96875,0.921875,0.50000,0.87500,0.656250,58.362331,261.373008,8192,2,1024,cuda,
6,qwen3_06b,"con reranker, sin filtro",0.65625,0.93750,0.778646,0.46875,0.78125,0.606771,115.540106,261.373008,8192,2,1024,cuda,cuda
7,qwen3_06b,"con reranker, filtro modulo con repeticion",0.84375,0.96875,0.901042,0.59375,0.84375,0.700521,109.825947,261.373008,8192,2,1024,cuda,cuda
8,qwen3_8b,"sin reranker, sin filtro",0.81250,0.96875,0.872396,0.53125,0.75000,0.622396,205.053428,1573.431029,8192,2,4096,cuda+cpu,
9,qwen3_8b,"sin reranker, filtro modulo con repeticion",0.90625,1.00000,0.940104,0.56250,0.87500,0.690104,197.777803,1573.431029,8192,2,4096,cuda+cpu,


## 11. Guardar los CSV

In [10]:
SALIDA_RESUMEN.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(SALIDA_RESUMEN, index=False)
por_pregunta.to_csv(SALIDA_PREGUNTA, index=False)
print("Guardado", SALIDA_RESUMEN.relative_to(RAIZ))
print("Guardado", SALIDA_PREGUNTA.relative_to(RAIZ))

Guardado data\resultados\modelos_embedding.csv
Guardado data\resultados\modelos_embedding_por_pregunta.csv


## 12. Criterio de RF-13 aplicado

Se aplica tal cual está escrito arriba, con `elegir_modelo(resumen, referencia="nomic")` sobre «con reranker, filtro modulo con repeticion». Los aciertos se pasan en número de preguntas sobre 32. `elegir_modelo` solo devuelve el nombre: la columna `cumple` de la tabla recalcula la misma regla (ev_hit@4 ≥ nomic + 2, hit@4 ≥ nomic, sin contar a nomic) para mostrarla, y no sale de esa función.

In [11]:
N = len(preguntas)
filas_criterio = [
    {
        "modelo": f["modelo"],
        "configuracion": f["configuracion"],
        "hit@4": round(f["hit@4"] * N),
        "ev_hit@4": round(f["ev_hit@4"] * N),
        "latencia_ms": f["latencia media consulta (ms)"],
    }
    for f in resumen.to_dict("records")
]
elegido = elegir_modelo(filas_criterio, referencia="nomic")
final = [f for f in filas_criterio if f["configuracion"] == "con reranker, filtro modulo con repeticion"]
base = next(f for f in final if f["modelo"] == "nomic")
tabla = pd.DataFrame(final).assign(
    cumple=lambda d: (d["ev_hit@4"] >= base["ev_hit@4"] + 2) & (d["hit@4"] >= base["hit@4"]) & (d["modelo"] != "nomic")
)
display(tabla)
print("Modelo elegido:", elegido)

,modelo,configuracion,hit@4,ev_hit@4,latencia_ms,cumple
0,nomic,"con reranker, filtro modulo con repeticion",31,28,105.693175,False
1,qwen3_06b,"con reranker, filtro modulo con repeticion",31,27,109.825947,False
2,qwen3_8b,"con reranker, filtro modulo con repeticion",32,28,256.058075,False
3,bge_m3,"con reranker, filtro modulo con repeticion",32,28,117.319606,False
4,embeddinggemma,"con reranker, filtro modulo con repeticion",32,27,151.233247,False
5,jina_code,"con reranker, filtro modulo con repeticion",31,25,105.999287,False


Modelo elegido: nomic
